# Exploración API Open-Meteo 
**Prueba Técnica — Parte 2 | Ingeniero de Datos**


In [2]:
# Verificar que están disponibles
import requests
import pandas as pd
import json

Probar con Medellín

In [3]:
URL = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude"     : 6.2442,
    "longitude"    : -75.5812,
    "current"      : "temperature_2m,weather_code",  # campos separados por coma
    "timezone"     : "America/Bogota",
    "forecast_days": 1
}

response = requests.get(URL, params=params, timeout=10)

# Siempre verificar el status antes de procesar
print(f"Status code : {response.status_code}")
print(f"URL final   : {response.url}")
response.raise_for_status()  # lanza excepción si es 4xx o 5xx

Status code : 200
URL final   : https://api.open-meteo.com/v1/forecast?latitude=6.2442&longitude=-75.5812&current=temperature_2m%2Cweather_code&timezone=America%2FBogota&forecast_days=1



## Explorar la estructura del JSON


In [4]:
data = response.json()

print(list(data.keys()))

['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'current_units', 'current']


In [5]:
print(data["current"].keys())

dict_keys(['time', 'interval', 'temperature_2m', 'weather_code'])


In [6]:
print(data["current"].values())

dict_values(['2026-03-02T18:45', 900, 25.8, 80])


In [7]:
print(json.dumps(data["current_units"]))

{"time": "iso8601", "interval": "seconds", "temperature_2m": "\u00b0C", "weather_code": "wmo code"}


\u00b0C es unicode para grados celcius

In [8]:
# Ver la respuesta completa formateada
print(json.dumps(data, indent=2))

{
  "latitude": 6.25,
  "longitude": -75.625,
  "generationtime_ms": 0.02753734588623047,
  "utc_offset_seconds": -18000,
  "timezone": "America/Bogota",
  "timezone_abbreviation": "GMT-5",
  "elevation": 1476.0,
  "current_units": {
    "time": "iso8601",
    "interval": "seconds",
    "temperature_2m": "\u00b0C",
    "weather_code": "wmo code"
  },
  "current": {
    "time": "2026-03-02T18:45",
    "interval": 900,
    "temperature_2m": 25.8,
    "weather_code": 80
  }
}


In [9]:
current = data["current"]

temperatura  = current["temperature_2m"]     # float, °C
weather_code = current["weather_code"]        # int, código WMO
timestamp    = current["time"]               # str ISO 8601 en hora local

print(timestamp)
print(temperatura)
print(weather_code)


2026-03-02T18:45
25.8
80


## Decodificar el `weather_code`   
https://open-meteo.com/en/docs#weather_variable_documentation

In [10]:
WMO_CODES = {
    0 : "Cielo despejado",
    1 : "Principalmente despejado",
    2 : "Parcialmente nublado",
    3 : "Nublado",
    45: "Niebla",
    48: "Niebla con escarcha depositante",
    51: "Llovizna ligera",
    53: "Llovizna moderada",
    55: "Llovizna densa",
    56: "Llovizna helada ligera",
    57: "Llovizna helada densa",
    61: "Lluvia ligera",
    63: "Lluvia moderada",
    65: "Lluvia fuerte",
    66: "Lluvia helada ligera",
    67: "Lluvia helada fuerte",
    71: "Nevada ligera",
    73: "Nevada moderada",
    75: "Nevada fuerte",
    77: "Granos de nieve",
    80: "Aguacero ligero",
    81: "Aguacero moderado",
    82: "Aguacero fuerte",
    85: "Nevada en aguacero ligera",
    86: "Nevada en aguacero fuerte",
    95: "Tormenta eléctrica",
    96: "Tormenta eléctrica con granizo ligero",
    99: "Tormenta eléctrica con granizo fuerte",
}


In [11]:

def decode_weather(code: int) -> str:
    """
    Convierte un código WMO al texto descriptivo en español.
    El fallback cubre casos donde Open-Meteo agregue códigos nuevos.
    """
    return WMO_CODES.get(int(code), f"Código desconocido ({code})")



print(f"Código {weather_code} : '{decode_weather(weather_code)}'")


Código 80 : 'Aguacero ligero'



##  10 municipios del Valle de Aburrá

In [12]:
# Archivo maestro — municipios del Área Metropolitana del Valle de Aburrá
MUNICIPIOS = [
    {"nombre": "Medellín",    "lat":  6.2442, "lon": -75.5812},
    {"nombre": "Envigado",    "lat":  6.1720, "lon": -75.5872},
    {"nombre": "Itagüí",      "lat":  6.1847, "lon": -75.5996},
    {"nombre": "La Estrella", "lat":  6.1564, "lon": -75.6433},
    {"nombre": "Bello",       "lat":  6.3367, "lon": -75.5569},
    {"nombre": "Copacabana",  "lat":  6.3488, "lon": -75.5100},
    {"nombre": "Sabaneta",    "lat":  6.1514, "lon": -75.6167},
    {"nombre": "Caldas",      "lat":  6.0940, "lon": -75.6371},
    {"nombre": "Girardota",   "lat":  6.3797, "lon": -75.4461},
    {"nombre": "Barbosa",     "lat":  6.4394, "lon": -75.3328},
]

print(f"Municipios en el maestro: {len(MUNICIPIOS)}")
pd.DataFrame(MUNICIPIOS)

Municipios en el maestro: 10


,nombre,lat,lon
0,Medellín,6.2442,-75.5812
1,Envigado,6.1720,-75.5872
2,Itagüí,6.1847,-75.5996
3,La Estrella,6.1564,-75.6433
4,Bello,6.3367,-75.5569
5,Copacabana,6.3488,-75.5100
6,Sabaneta,6.1514,-75.6167
7,Caldas,6.0940,-75.6371
8,Girardota,6.3797,-75.4461
9,Barbosa,6.4394,-75.3328


In [13]:
def fetch_municipio(session: requests.Session, municipio: dict) -> dict:
    """
    Consulta el clima actual de un municipio.
    Retorna un dict con todos los campos listos para insertar en la BD.
    """
    params = {
        "latitude"     : municipio["lat"],
        "longitude"    : municipio["lon"],
        "current"      : "temperature_2m,weather_code",
        "timezone"     : "America/Bogota",
        "forecast_days": 1,
    }

    resp = session.get(URL, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    cur  = data["current"]
    code = int(cur["weather_code"])

    return {
        "municipio"           : municipio["nombre"],
        "latitud"             : round(data["latitude"], 4),
        "longitud"            : round(data["longitude"], 4),
        "temperatura_c"       : round(cur["temperature_2m"], 1),
        "weather_code"        : code,
        "descripcion_clima"   : decode_weather(code),
        "ultima_actualizacion": cur["time"],  # ISO 8601 en hora local
    }




In [14]:
resultados = []

with requests.Session() as session:
    for mun in MUNICIPIOS:
        try:
            fila = fetch_municipio(session, mun)
            resultados.append(fila)

            print(mun["nombre"], fila["temperatura_c"], fila["descripcion_clima"])

        except requests.RequestException as e:
            print(mun["nombre"], "Error:", e)

print("Total obtenidos:", len(resultados))

Medellín 25.8 Aguacero ligero
Envigado 24.3 Aguacero ligero
Itagüí 24.4 Aguacero ligero
La Estrella 23.0 Aguacero ligero
Bello 22.7 Nublado
Copacabana 22.9 Nublado
Sabaneta 24.1 Aguacero ligero
Caldas 23.1 Aguacero ligero
Girardota 23.1 Nublado
Barbosa 19.3 Nublado
Total obtenidos: 10


In [15]:
# Convertir a DataFrame para inspección
df = pd.DataFrame(resultados)
df

,municipio,latitud,longitud,temperatura_c,weather_code,descripcion_clima,ultima_actualizacion
0,Medellín,6.250,-75.625,25.8,80,Aguacero ligero,2026-03-02T18:45
1,Envigado,6.125,-75.625,24.3,80,Aguacero ligero,2026-03-02T18:45
2,Itagüí,6.125,-75.625,24.4,80,Aguacero ligero,2026-03-02T18:45
3,La Estrella,6.125,-75.625,23.0,80,Aguacero ligero,2026-03-02T18:45
4,Bello,6.375,-75.500,22.7,3,Nublado,2026-03-02T18:45
5,Copacabana,6.375,-75.500,22.9,3,Nublado,2026-03-02T18:45
6,Sabaneta,6.125,-75.625,24.1,80,Aguacero ligero,2026-03-02T18:45
7,Caldas,6.125,-75.625,23.1,80,Aguacero ligero,2026-03-02T18:45
8,Girardota,6.375,-75.500,23.1,3,Nublado,2026-03-02T18:45
9,Barbosa,6.500,-75.375,19.3,3,Nublado,2026-03-02T18:45


In [16]:
print("Tipos de datos:")
print(df.dtypes)
print(f"\nShape: {df.shape}")
print(f"Nulos: {df.isnull().sum().sum()}")

Tipos de datos:
municipio                object
latitud                 float64
longitud                float64
temperatura_c           float64
weather_code              int64
descripcion_clima        object
ultima_actualizacion     object
dtype: object

Shape: (10, 7)
Nulos: 0



## simular la lógica UPSERT

In [17]:
# Simular BD existente con datos viejos
db_simulada = df.copy()
db_simulada["temperatura_c"]      = 99.9
db_simulada["descripcion_clima"]  = "Dato viejo"
db_simulada["ultima_actualizacion"] = "2020-01-01T00:00"

print("Estado ANTES del upsert:")
print(db_simulada[["municipio", "temperatura_c", "ultima_actualizacion"]]
      .to_string(index=False))

Estado ANTES del upsert:
  municipio  temperatura_c ultima_actualizacion
   Medellín           99.9     2020-01-01T00:00
   Envigado           99.9     2020-01-01T00:00
     Itagüí           99.9     2020-01-01T00:00
La Estrella           99.9     2020-01-01T00:00
      Bello           99.9     2020-01-01T00:00
 Copacabana           99.9     2020-01-01T00:00
   Sabaneta           99.9     2020-01-01T00:00
     Caldas           99.9     2020-01-01T00:00
  Girardota           99.9     2020-01-01T00:00
    Barbosa           99.9     2020-01-01T00:00


In [18]:
CAMPOS_CLIMA = ["temperatura_c", "weather_code", "descripcion_clima", "ultima_actualizacion"]

def upsert(db: pd.DataFrame, nuevos: pd.DataFrame) -> pd.DataFrame:
    """
    Upsert por nombre de municipio (clave primaria).
    - Existe  : actualiza solo campos climáticos + fecha
    - No existe : inserta fila completa
    """
    db = db.set_index("municipio").copy()
    nuevos = nuevos.set_index("municipio")

    for municipio, row in nuevos.iterrows():
        if municipio in db.index:
            db.loc[municipio, CAMPOS_CLIMA] = row[CAMPOS_CLIMA]  # UPDATE
        else:
            db.loc[municipio] = row                              # INSERT

    return db.reset_index()


db_actualizada = upsert(db_simulada, df)

print("Estado DESPUÉS del upsert:")
print(db_actualizada[["municipio", "temperatura_c", "ultima_actualizacion"]]
      .to_string(index=False))

Estado DESPUÉS del upsert:
  municipio  temperatura_c ultima_actualizacion
   Medellín           25.8     2026-03-02T18:45
   Envigado           24.3     2026-03-02T18:45
     Itagüí           24.4     2026-03-02T18:45
La Estrella           23.0     2026-03-02T18:45
      Bello           22.7     2026-03-02T18:45
 Copacabana           22.9     2026-03-02T18:45
   Sabaneta           24.1     2026-03-02T18:45
     Caldas           23.1     2026-03-02T18:45
  Girardota           23.1     2026-03-02T18:45
    Barbosa           19.3     2026-03-02T18:45


## sentencias SQL del UPSERT

In [19]:
def generar_sql_upsert(df: pd.DataFrame) -> list[str]:
    """Genera sentencias INSERT ... ON CONFLICT para PostgreSQL."""
    stmts = []
    for _, row in df.iterrows():
        sql = f"""INSERT INTO clima_municipios
    (municipio, latitud, longitud, temperatura_c, weather_code, descripcion_clima, ultima_actualizacion)
VALUES
    ('{row.municipio}', {row.latitud}, {row.longitud}, {row.temperatura_c}, {row.weather_code}, '{row.descripcion_clima}', '{row.ultima_actualizacion}')
ON CONFLICT (municipio)
DO UPDATE SET
    temperatura_c        = EXCLUDED.temperatura_c,
    weather_code         = EXCLUDED.weather_code,
    descripcion_clima    = EXCLUDED.descripcion_clima,
    ultima_actualizacion = EXCLUDED.ultima_actualizacion;"""
        stmts.append(sql)
    return stmts


sentencias = generar_sql_upsert(df)

for stmt in sentencias:
    print(stmt)
    print()

INSERT INTO clima_municipios
    (municipio, latitud, longitud, temperatura_c, weather_code, descripcion_clima, ultima_actualizacion)
VALUES
    ('Medellín', 6.25, -75.625, 25.8, 80, 'Aguacero ligero', '2026-03-02T18:45')
ON CONFLICT (municipio)
DO UPDATE SET
    temperatura_c        = EXCLUDED.temperatura_c,
    weather_code         = EXCLUDED.weather_code,
    descripcion_clima    = EXCLUDED.descripcion_clima,
    ultima_actualizacion = EXCLUDED.ultima_actualizacion;

INSERT INTO clima_municipios
    (municipio, latitud, longitud, temperatura_c, weather_code, descripcion_clima, ultima_actualizacion)
VALUES
    ('Envigado', 6.125, -75.625, 24.3, 80, 'Aguacero ligero', '2026-03-02T18:45')
ON CONFLICT (municipio)
DO UPDATE SET
    temperatura_c        = EXCLUDED.temperatura_c,
    weather_code         = EXCLUDED.weather_code,
    descripcion_clima    = EXCLUDED.descripcion_clima,
    ultima_actualizacion = EXCLUDED.ultima_actualizacion;

INSERT INTO clima_municipios
    (municipio, lati